In [7]:
import pandas as pd
import numpy as np
import csv, time
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeMarrakesh
# from qiskit_ibm_runtime import EstimatorV2 as Estimator
# from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
# from qiskit.quantum_info import SparsePauliOp

# QRU Circuit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, Session, EstimatorV2 as Estimator

# ── 設定 ──────────────────────────────────────────────────────
CALIB_CSV   = 'qru_otc_multi_obs_20260503_205659.csv'
OUT_CSV     = f'qru_otc_multi_ibm_marrakesh_calib_{time.strftime("%Y%m%d_%H%M%S")}.csv'
TARGET_VALS = [-0.8, -0.4, 0.4, 0.8]   # 精簡版只用 ±0.4, ±0.8
N_REPEAT    = 3              # 每組電路重複 3 次
N_SHOTS     = 1024

# ── Observable 分組（和 SV 資料庫一致）───────────────────────
OBS_GROUPS = [
    [(0, 'Z', 'Z0',  0),
     (4, 'Z', 'Z4',  8),
     (6, 'Z', 'Z6', 12),
     (5, 'X', 'X5', 11),
     (7, 'X', 'X7', 15)],
    [(1, 'Z', 'Z1',  2),
     (5, 'Z', 'Z5', 10),
     (7, 'Z', 'Z7', 14),
     (4, 'X', 'X4',  9),
     (6, 'X', 'X6', 13)],
]




In [ ]:
# ── Real backend 設定 ─────────────────────────────────────────
# 學校帳號
service = QiskitRuntimeService(name="NTU")
real_backend = service.backend("ibm_marrakesh", instance="<YOUR_INSTANCE_NAME>")
# real_backend = service.backend("ibm_aachen", instance="<YOUR_INSTANCE_NAME>")

pm           = generate_preset_pass_manager(backend=real_backend, optimization_level=1)

In [ ]:
# ── Fake backend 設定 ─────────────────────────────────────────
fake_backend = AerSimulator.from_backend(FakeMarrakesh())
pm           = generate_preset_pass_manager(backend=fake_backend, optimization_level=1)



In [9]:
# ── Observable 建構 ───────────────────────────────────────────
def make_obs(qubit, pauli, n=10):
    s = ['I'] * n
    s[n-1-qubit] = pauli
    return SparsePauliOp(''.join(s))

# ── 讀取 SV 資料庫 ────────────────────────────────────────────
param_cols = [c for c in pd.read_csv(CALIB_CSV, nrows=0).columns if c.startswith('p')]
calib_df   = pd.read_csv(CALIB_CSV)

# 只取 target_val = ±0.8
calib_sub = calib_df[calib_df['target_val'].isin(TARGET_VALS)].copy()
print(f"使用筆數: {len(calib_sub)} (target_val={TARGET_VALS})")


使用筆數: 400 (target_val=[-0.8, -0.4, 0.4, 0.8])


In [10]:
def build_qru_circuit(data_input, hidden_input, params):
    """
    data_input:   numpy array, shape (8,)
    hidden_input: numpy array, shape (8,), 初始為全零
    params:       numpy array, shape (131,)
    """

    data   = QuantumRegister(4, 'data')
    hidden = QuantumRegister(4, 'hidden')
    anc    = QuantumRegister(2, 'anc')
    qc = QuantumCircuit(data, hidden, anc)

    # ==================
    # 1. ENCODING
    # ==================
    # Data: params[0:16], 每 qubit 4 個參數
    qc.barrier(label='ENC')
    for i in range(4):
        rx_angle = data_input[2*i] * params[4*i]     + params[4*i + 1]
        ry_angle = data_input[2*i + 1] * params[4*i + 2] + params[4*i + 3]
        qc.rx(rx_angle, data[i])
        qc.ry(ry_angle, data[i])

    # Hidden: params[16:32]
    for i in range(4):
        rx_angle = hidden_input[2*i] * params[16 + 4*i]     + params[16 + 4*i + 1]
        ry_angle = hidden_input[2*i + 1] * params[16 + 4*i + 2] + params[16 + 4*i + 3]
        qc.rx(rx_angle, hidden[i])
        qc.ry(ry_angle, hidden[i])

    # 循環 CX on data: 0→1→2→3→0
    for i in range(4):
        qc.cx(data[i], data[(i+1) % 4])

    # 循環 CX on hidden: 4→5→6→7→4
    for i in range(4):
        qc.cx(hidden[i], hidden[(i+1) % 4])

    # ==================
    # 2. UPDATE & RESET
    # ==================
    qc.barrier(label='UPD')
    qc.cx(data[0], hidden[0])
    qc.cswap(hidden[0], hidden[1], anc[0])

    # Rot on hidden[0]: params[32:35]
    qc.rz(params[32], hidden[0])
    qc.ry(params[33], hidden[0])
    qc.rz(params[34], hidden[0])

    qc.cswap(hidden[0], hidden[3], anc[1])
    # ancilla 保持 |0⟩，不需 reset

    # ==================
    # 3. VARIATIONAL LAYERS（4 層）
    # ==================
    idx = 35
    for layer in range(4):
        qc.barrier(label=f'VAR{layer+1}')
        # Rot on all non-ancilla qubits
        for reg in [data, hidden]:
            for i in range(4):
                qc.rz(params[idx],     reg[i]); idx += 1
                qc.ry(params[idx],     reg[i]); idx += 1
                qc.rz(params[idx],     reg[i]); idx += 1

        # 循環 CX on all non-ancilla: data[0..3] → hidden[0..3] → back
        all_q = [data[i] for i in range(4)] + [hidden[i] for i in range(4)]
        for i in range(8):
            qc.cx(all_q[i], all_q[(i+1) % 8])

    return qc

In [11]:
# ── 寫 CSV header ─────────────────────────────────────────────
with open(OUT_CSV, 'w', newline='') as f:
    csv.writer(f).writerow(
        ['group_id', 'obs_name', 'target_val', 'rep_id', 'sv_val', 'fake_val']
    )

# ── 主迴圈 ────────────────────────────────────────────────────
st    = time.time()
# 每個 (group, target_val, db_rep) 組合對應一個電路，跑 N_REPEAT 次
# group 0: 5 obs, group 1: 5 obs
# 每個 group × 2 target_val × 10 db_reps = 20 個不同電路
# 但精簡版只取 db_rep_id=0（第一個 rep），跑 N_REPEAT 次
# 共 2 groups × 2 target_val × 1 db_rep = 4 個電路，各跑 3 次 = 12 jobs

done  = 0
total = 2 * len(TARGET_VALS) * N_REPEAT  # 12

for grp_id, group in enumerate(OBS_GROUPS):
    obs_list = [(q, p, name) for q, p, name, _ in group]

    for target_val in TARGET_VALS:
        # 取 rep_id=0 的電路參數
        mask = (
            (calib_sub['group_id'] == grp_id) &
            (calib_sub['target_val'] == target_val) &
            (calib_sub['rep_id'] == 0)
        )
        rows = calib_sub[mask]
        if len(rows) == 0:
            print(f"Warning: 找不到 group={grp_id} target={target_val} rep=0")
            continue

        # 取第一個 observable 的 params（同組共用同一組 params）
        row      = rows.iloc[0]
        params   = row[param_cols].values.astype(float)
        sv_vals  = {r['obs_name']: r['sv_val'] for _, r in rows.iterrows()}

        # 建電路
        qc     = build_qru_circuit(np.zeros(8), np.zeros(8), params)
        isa_qc = pm.run(qc)

        # 建所有 observable
        obs_objs = [make_obs(q, p) for q, p, _ in obs_list]
        isa_obs  = [o.apply_layout(isa_qc.layout) for o in obs_objs]

        # 重複 N_REPEAT 次
        for rep in range(N_REPEAT):
            estimator = Estimator(mode=real_backend)
            estimator.options.default_shots = N_SHOTS
            evs = estimator.run([(isa_qc, isa_obs)]).result()[0].data.evs

            # 寫入每個 observable 的結果
            with open(OUT_CSV, 'a', newline='') as f:
                writer = csv.writer(f)
                for i, (q, p, obs_name) in enumerate(obs_list):
                    writer.writerow([
                        grp_id, obs_name, target_val, rep,
                        sv_vals.get(obs_name, float('nan')),
                        float(evs[i])
                    ])

            done += 1
            elapsed = time.time() - st
            eta     = elapsed / done * (total - done) / 60
            print(f"[{done:>2}/{total}] grp={grp_id} tv={target_val:+.1f} rep={rep} | "
                  f"evs={[f'{v:.3f}' for v in evs]} | "
                  f"{elapsed/60:.1f}min ETA={eta:.1f}min")

print(f"\n完成！儲存至 {OUT_CSV}")



[ 1/24] grp=0 tv=-0.8 rep=0 | evs=['-0.140', '-0.081', '-0.141', '-0.249', '-0.303'] | 0.4min ETA=8.2min
[ 2/24] grp=0 tv=-0.8 rep=1 | evs=['-0.061', '-0.120', '-0.091', '-0.185', '-0.312'] | 0.7min ETA=7.3min
[ 3/24] grp=0 tv=-0.8 rep=2 | evs=['-0.111', '-0.118', '-0.157', '-0.206', '-0.284'] | 0.9min ETA=6.6min
[ 4/24] grp=0 tv=-0.4 rep=0 | evs=['-0.087', '-0.065', '-0.103', '-0.002', '-0.213'] | 1.2min ETA=6.2min
[ 5/24] grp=0 tv=-0.4 rep=1 | evs=['-0.055', '-0.075', '-0.083', '-0.110', '-0.186'] | 1.5min ETA=5.8min
[ 6/24] grp=0 tv=-0.4 rep=2 | evs=['-0.084', '-0.089', '-0.121', '-0.085', '-0.198'] | 1.8min ETA=5.4min
[ 7/24] grp=0 tv=+0.4 rep=0 | evs=['0.083', '0.055', '0.108', '0.071', '0.118'] | 2.1min ETA=5.2min
[ 8/24] grp=0 tv=+0.4 rep=1 | evs=['0.077', '0.103', '0.051', '0.063', '0.133'] | 2.4min ETA=4.8min
[ 9/24] grp=0 tv=+0.4 rep=2 | evs=['0.018', '0.040', '0.132', '0.123', '0.059'] | 2.7min ETA=4.5min
[10/24] grp=0 tv=+0.8 rep=0 | evs=['0.161', '0.172', '0.120', '0.200',

In [8]:
# ── 立即分析：估計 alpha 和 K ─────────────────────────────────
print(f"\n=== Alpha 和 K 估計（精簡版）===")
result_df = pd.read_csv(OUT_CSV)

obs_all = ['Z0','Z1','Z4','Z5','Z6','X4','X5','X6','X7','Z7']
print(f"{'obs':<6} {'alpha':>8} {'K':>8} {'n':>4}")
print("-"*30)
for obs in sorted(obs_all):
    sub = result_df[result_df['obs_name']==obs]
    if len(sub) == 0:
        continue
    sv   = sub['sv_val'].values
    fk   = sub['fake_val'].values
    mask = np.abs(sv) > 0.05
    if mask.sum() < 2:
        continue
    alpha  = np.dot(sv[mask], fk[mask]) / np.dot(sv[mask], sv[mask])
    mu     = alpha * sv
    res    = fk - mu
    factor = np.sqrt(np.maximum(1 - mu**2, 1e-6)).mean()
    K      = res.std() / factor
    print(f"{obs:<6} {alpha:>8.4f} {K:>8.4f} {len(sub):>4}")


=== Alpha 和 K 估計（精簡版）===
obs       alpha        K    n
------------------------------
X4       0.6670   0.0445   12
X5       0.7034   0.0600   12
X6       0.6665   0.0291   12
X7       0.6925   0.1005   12
Z0       0.5836   0.0621   12
Z1       0.6424   0.0618   12
Z4       0.5779   0.0613   12
Z5       0.5797   0.0356   12
Z6       0.5662   0.0533   12
Z7       0.5256   0.0411   12
